In [1]:
# Locate the repository when running from any notebook subfolder.
from pathlib import Path
import os
_notebook_start = Path.cwd().resolve()
_notebook_root = next(
    (p for p in [_notebook_start, *_notebook_start.parents]
     if (p / "docs" / "predictor_list.csv").is_file()
     and (p / "notebooks").is_dir()), None
)
if _notebook_root is None:
    raise FileNotFoundError("Cannot locate the project repository.")
os.chdir(_notebook_root)


In [2]:
# Purpose: Load the official dataset and check order identifiers.

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

input_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "research_data_weather_price_holiday_assume_utc.csv"
)

df = pd.read_csv(
    input_path,
    dtype={"order_id": "string"},
    low_memory=False
)

df["purchase_time"] = pd.to_datetime(
    df["order_purchase_timestamp"],
    errors="coerce"
)

df["is_late"] = pd.to_numeric(
    df["is_late"],
    errors="coerce"
)

assert df["order_id"].notna().all()
assert df["order_id"].is_unique
assert df["purchase_time"].notna().all()

print("Total orders:", len(df))
print("Valid labels:", df["is_late"].isin([0, 1]).sum())

Total orders: 99441
Valid labels: 96470


In [3]:
# Purpose: Find approximate 70/20/10 chronological boundaries.

labeled = (
    df.loc[df["is_late"].isin([0, 1])]
    .copy()
    .sort_values(["purchase_time", "order_id"])
    .reset_index(drop=True)
)

labeled["purchase_date"] = labeled["purchase_time"].dt.normalize()

daily_counts = (
    labeled.groupby("purchase_date")
    .size()
    .sort_index()
)

cumulative_share = daily_counts.cumsum() / len(labeled)

train_last_date = cumulative_share.ge(0.70).idxmax()
validation_last_date = cumulative_share.ge(0.90).idxmax()

validation_start = train_last_date + pd.Timedelta(days=1)
test_start = validation_last_date + pd.Timedelta(days=1)

print("Train ends:", train_last_date.date())
print("Validation starts:", validation_start.date())
print("Validation ends:", validation_last_date.date())
print("Test starts:", test_start.date())

Train ends: 2018-04-15
Validation starts: 2018-04-16
Validation ends: 2018-07-18
Test starts: 2018-07-19


In [4]:
# Purpose: Assign candidate splits and inspect class distribution.

labeled["split"] = "test"

labeled.loc[
    labeled["purchase_time"] < test_start,
    "split"
] = "validation"

labeled.loc[
    labeled["purchase_time"] < validation_start,
    "split"
] = "train"

split_summary = (
    labeled.groupby("split")
    .agg(
        orders=("order_id", "size"),
        late_orders=("is_late", "sum"),
        earliest_purchase=("purchase_time", "min"),
        latest_purchase=("purchase_time", "max")
    )
    .reindex(["train", "validation", "test"])
)

split_summary["not_late_orders"] = (
    split_summary["orders"]
    - split_summary["late_orders"]
)

split_summary["share_percent"] = (
    split_summary["orders"] / len(labeled) * 100
).round(2)

split_summary["late_rate_percent"] = (
    split_summary["late_orders"]
    / split_summary["orders"] * 100
).round(2)

display(split_summary)

,orders,late_orders,earliest_purchase,latest_purchase,not_late_orders,share_percent,late_rate_percent
split,,,,,,,
train,67598,5291.0,2016-09-15 12:16:38,2018-04-15 23:55:50,62307.0,70.07,7.83
validation,19427,739.0,2018-04-16 00:03:57,2018-07-18 23:58:18,18688.0,20.14,3.80
test,9445,504.0,2018-07-19 00:27:48,2018-08-29 15:00:37,8941.0,9.79,5.34


In [5]:
# Purpose: Audit delivery-label availability at candidate boundaries.

labeled["delivery_time"] = pd.to_datetime(
    labeled["order_delivered_customer_date"],
    errors="coerce"
)

audit_rows = []

for split_name, cutoff in [
    ("train", validation_start),
    ("validation", test_start)
]:
    sample = labeled.loc[labeled["split"].eq(split_name)]

    delivered_before = sample["delivery_time"].lt(cutoff)

    audit_rows.append({
        "split": split_name,
        "next_stage_start": cutoff,
        "orders": len(sample),
        "delivered_before_cutoff": int(delivered_before.sum()),
        "not_delivered_before_cutoff": int((~delivered_before).sum()),
        "late_orders_not_delivered_before_cutoff": int(
            (
                ~delivered_before
                & sample["is_late"].eq(1)
            ).sum()
        )
    })

label_audit = pd.DataFrame(audit_rows)

display(label_audit)

,split,next_stage_start,orders,delivered_before_cutoff,not_delivered_before_cutoff,late_orders_not_delivered_before_cutoff
0,train,2018-04-16,67598,64430,3168,903
1,validation,2018-07-19,19427,17647,1780,142


In [6]:
# Purpose: Audit fixed maturation gaps before the next stage.

gap_results = []

for gap_days in [30, 45]:
    for split_name, cutoff in [
        ("train", validation_start),
        ("validation", test_start)
    ]:
        original = labeled.loc[
            labeled["split"].eq(split_name)
        ].copy()

        purchase_end = cutoff - pd.Timedelta(days=gap_days)

        # Exclude recent purchases by a fixed date rule.
        candidate = original.loc[
            original["purchase_time"] < purchase_end
        ].copy()

        available = candidate["delivery_time"].lt(cutoff)

        gap_results.append({
            "split": split_name,
            "gap_days": gap_days,
            "purchase_end_exclusive": purchase_end,
            "label_cutoff": cutoff,
            "candidate_orders": len(candidate),
            "orders_in_gap": len(original) - len(candidate),
            "labels_available": int(available.sum()),
            "labels_pending": int((~available).sum()),
            "late_labels_available": int(
                (
                    available
                    & candidate["is_late"].eq(1)
                ).sum()
            ),
            "late_labels_pending": int(
                (
                    ~available
                    & candidate["is_late"].eq(1)
                ).sum()
            )
        })

gap_audit = pd.DataFrame(gap_results)

display(gap_audit)

,split,gap_days,purchase_end_exclusive,label_cutoff,candidate_orders,orders_in_gap,labels_available,labels_pending,late_labels_available,late_labels_pending
0,train,30,2018-03-17,2018-04-16,61032,6566,60598,434,4295,413
1,validation,30,2018-06-19,2018-07-19,13944,5483,13913,31,593,27
2,train,45,2018-03-02,2018-04-16,57585,10013,57449,136,3720,136
3,validation,45,2018-06-04,2018-07-19,10783,8644,10776,7,567,7


In [7]:
# Purpose: Assign chronological splits with a 45-day maturation gap.

labeled["final_split"] = "excluded"

train_mask = (
    labeled["purchase_time"].lt("2018-03-02")
    & labeled["delivery_time"].lt("2018-04-16")
)

validation_mask = (
    labeled["purchase_time"].ge("2018-04-16")
    & labeled["purchase_time"].lt("2018-06-04")
    & labeled["delivery_time"].lt("2018-07-19")
)

validation_a_mask = (
    validation_mask
    & labeled["purchase_time"].lt("2018-05-16")
)

validation_b_mask = (
    validation_mask
    & labeled["purchase_time"].ge("2018-05-16")
)

test_mask = labeled["split"].eq("test")

labeled.loc[train_mask, "final_split"] = "train"
labeled.loc[validation_a_mask, "final_split"] = "validation_a"
labeled.loc[validation_b_mask, "final_split"] = "validation_b"
labeled.loc[test_mask, "final_split"] = "test"

split_order = [
    "train",
    "validation_a",
    "validation_b",
    "test"
]

selected = labeled.loc[
    labeled["final_split"].isin(split_order)
].copy()

final_summary = (
    selected.groupby("final_split")
    .agg(
        orders=("order_id", "size"),
        late_orders=("is_late", "sum"),
        earliest_purchase=("purchase_time", "min"),
        latest_purchase=("purchase_time", "max")
    )
    .reindex(split_order)
)

final_summary["share_percent"] = (
    final_summary["orders"] / len(selected) * 100
).round(2)

final_summary["late_rate_percent"] = (
    final_summary["late_orders"]
    / final_summary["orders"] * 100
).round(2)

display(final_summary)

print("Selected orders:", len(selected))
print(
    "Excluded labeled orders:",
    labeled["final_split"].eq("excluded").sum()
)

,orders,late_orders,earliest_purchase,latest_purchase,share_percent,late_rate_percent
final_split,,,,,,
train,57449,3720.0,2016-09-15 12:16:38,2018-03-01 23:56:42,73.97,6.48
validation_a,7661,387.0,2018-04-16 00:03:57,2018-05-15 23:56:01,9.86,5.05
validation_b,3115,180.0,2018-05-16 00:05:40,2018-06-03 23:56:23,4.01,5.78
test,9445,504.0,2018-07-19 00:27:48,2018-08-29 15:00:37,12.16,5.34


Selected orders: 77670
Excluded labeled orders: 18800


In [8]:
# Purpose: Export the split manifest and summary.

output_dir = PROJECT_ROOT / "data" / "ml_preparation"
output_dir.mkdir(parents=True, exist_ok=True)

# Record why labeled orders are excluded.
labeled["exclusion_reason"] = ""

in_gap = (
    labeled["purchase_time"].ge("2018-03-02")
    & labeled["purchase_time"].lt("2018-04-16")
) | (
    labeled["purchase_time"].ge("2018-06-04")
    & labeled["purchase_time"].lt("2018-07-19")
)

excluded = labeled["final_split"].eq("excluded")

labeled.loc[
    excluded & in_gap,
    "exclusion_reason"
] = "Purchase falls within maturation gap"

labeled.loc[
    excluded & ~in_gap,
    "exclusion_reason"
] = "Delivery label unavailable at cutoff"

manifest = labeled[
    [
        "order_id",
        "purchase_time",
        "final_split",
        "exclusion_reason"
    ]
].rename(columns={"final_split": "split"})

assert manifest["order_id"].is_unique
assert manifest.loc[
    manifest["split"].eq("excluded"),
    "exclusion_reason"
].ne("").all()

manifest.to_csv(
    output_dir / "split_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

final_summary.to_csv(
    output_dir / "split_summary.csv",
    index=True,
    encoding="utf-8-sig"
)

print("Saved to:", output_dir)

Saved to: C:\Users\LENOVO\OneDrive\Desktop\PTTK\data\ml_preparation
